# GameTheory-13e : Réviser un modèle d'adversaire -- changer d'avis en perdant le moins possible

**Navigation** : [<< 13d-Optimistic-CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) | [Index](README.md)

**Kernel** : Python 3 (`coursia-ml-training`)

***

## Concept

Dans un jeu à information imparfaite, un agent borné porte des **croyances** sur son adversaire :
ce qu'il a observé, ce qu'il croit savoir de son comportement. Puis une évidence nouvelle arrive
qui **contredit** ces croyances -- l'adversaire vient de faire exactement ce que la base affirme
qu'il ne fait jamais. Trois attitudes sont possibles :

- **refuser** l'évidence et garder la base intacte : l'agent agit sur un modèle faux ;
- **tout réinitialiser** et repartir d'une croyance vide : l'information accumulée est perdue ;
- **réviser** : accepter l'évidence en renonçant au *minimum* de ce qu'il croyait.

La théorie AGM (Alchourron, Gardenfors, Makinson, 1985) formalise la troisième attitude --
la triade expansion / contraction / révision. La famille 13 l'a déjà croisée sans la nommer :
le safe subgame solving de 13b *répare* un blueprint sans détruire l'équilibre global ;
l'abstraction à dette bornée du 19 contrôle ce qu'une simplification peut coûter. Ici le geste
est épistémique : c'est la **croyance** que l'on contracte, pas la stratégie.

Ce carnet atteste l'opération *réviser une croyance* sur un substrat indépendant de l'attestation
logique classique ([Tweety-04-Belief-Revision](../SymbolicAI/Tweety/Tweety-04-Belief-Revision-Python.ipynb) :
opérateurs CrMas sur machine virtuelle Java). Ici, pas de JVM : la base de croyances décrit un
**adversaire dans un jeu répété**, la consistance et les noyaux sont décidés par le solveur SMT
**Z3**, l'ordre d'enracinement est **mesuré** par valeur prédictive sur une histoire tenue à part,
et les propriétés de la révision sont **testées sur une batterie** -- jamais simplement assertées.

Le témoin visé, en trois morceaux mesurés :

1. la **trace de révision** -- quels noyaux, quelle incision, quels atomes survivent ;
2. le **coût des attitudes rivales** -- refuser et réinitialiser, contre le type vrai, en gain cumulé ;
3. le **contrôle négatif** -- une évidence cohérente avec la base doit produire une expansion pure,
   c'est-à-dire zéro incision.


In [1]:
import itertools
from typing import Dict, List, Tuple, Optional, Set

import numpy as np
import z3

RNG = np.random.default_rng(seed=20261010)
print(f"numpy={np.__version__}  z3={z3.get_version_string()}")

numpy=2.4.4  z3=4.16.0


### Environnement opérationnel

Le solveur Z3 (`z3-solver`) est l'organe de consistance : chaque contrôle de cohérence et chaque
noyau passe par lui. Les intersections d'ensembles de types qui apparaissent ensuite servent
uniquement à *interpréter* ce que Z3 décide -- la décision reste au solveur.

## Section 1 -- Le banc : un jeu répété et une base de croyances sur l'adversaire

Un jeu répété à trois actions pour l'agent (*Sécuriser*, *Milieu*, *Attaquer*) et trois pour
l'adversaire, à somme nulle. L'adversaire est l'un de **huit types** comportementaux -- chaque
type est une politique qui réagit à l'historique des coups de l'agent. L'agent ne connaît pas
le type : il porte une **base de croyances** `B`, un ensemble fini d'atomes, chaque atome étant
une contrainte sur le type caché, héritée d'une observation passée ou d'un a priori théorique.
Les tours sont simultanés : la réponse adverse du tour `t` ne voit que les tours strictement
antérieurs.

In [2]:
# --- Le jeu : gain de l'agent (lignes = action de l'agent, colonnes = action de l'adversaire)
GAIN_AGENT = np.array([
    [ 1,  2, -1],   # Securiser : prudent partout, jamais excellent
    [ 3,  1, -4],   # Milieu    : exploite la prudence, piege complet face a l'attaque
    [-2,  0,  2],   # Attaquer  : puni par la prudence, tient face a l'attaque
])
ACTIONS = ('Securiser', 'Milieu', 'Attaquer')
N_ACTIONS = 3

def br_adverse(action_agent):
    """Meilleure reponse adverse a une action donnee de l'agent (somme nulle)."""
    return int(np.argmin(GAIN_AGENT[action_agent, :]))

def br_defaut():
    """Reponse adverse par defaut : minimiser le gain moyen de l'agent (agent uniforme)."""
    return int(np.argmin(GAIN_AGENT.mean(axis=0)))

# --- Les huit types adverses : chacun est une politique (fonction de l'historique agent)
def pol_br_frequent(hist):
    # meilleure reponse a l'action la plus FREQUENTE de l'agent
    if not hist:
        return br_defaut()
    counts = np.bincount(hist, minlength=N_ACTIONS)
    return br_adverse(int(np.argmax(counts)))

def pol_punisseur(hist):
    # prudence tant que l'agent n'attaque pas ; punition permanente ensuite
    return 2 if 2 in hist else 0

def pol_toujours_attaque(hist):
    return 2

def pol_bluffeur(hist):
    # attaque uniquement en reponse a un Milieu de l'agent au tour precedent
    return 2 if (hist and hist[-1] == 1) else 0

def pol_aleatoire(hist):
    # explicitement stochastique (tire dans la RNG globale du banc)
    return int(RNG.integers(N_ACTIONS))

def pol_miroir(hist):
    return hist[-1] if hist else 0

def pol_opportuniste(hist):
    # meilleure reponse a la DERNIERE action de l'agent
    return br_adverse(hist[-1]) if hist else br_defaut()

def pol_cycle(hist):
    return len(hist) % 3

def pol_bluffeur_retarde(hist):
    # prudence les cinq premiers tours, puis attaque en reponse au Milieu
    if len(hist) < 5:
        return 0
    return 2 if hist[-1] == 1 else 0

TYPES: Dict[str, Tuple[str, object]] = {
    't1': ('BR-frequent', pol_br_frequent),
    't2': ('Punisseur', pol_punisseur),
    't3': ('Toujours-attaque', pol_toujours_attaque),
    't4': ('Bluffeur', pol_bluffeur),
    't5': ('Aleatoire-uniforme', pol_aleatoire),
    't6': ('Opportuniste', pol_opportuniste),
    't7': ('Cycle-0-1-2', pol_cycle),
    't8': ('Bluffeur-retarde', pol_bluffeur_retarde),
}
NOMS_TYPES = list(TYPES)
print(f"{len(TYPES)} types adverses :")
for nom, (label, _) in TYPES.items():
    print(f"  {nom} : {label}")

8 types adverses :
  t1 : BR-frequent
  t2 : Punisseur
  t3 : Toujours-attaque
  t4 : Bluffeur
  t5 : Aleatoire-uniforme
  t6 : Opportuniste
  t7 : Cycle-0-1-2
  t8 : Bluffeur-retarde


In [3]:
# --- Sondes comportementales : les proprietes des types sont MESUREES sur les politiques,
#     jamais declarees a la main. Les sondes deterministes utilisent des histories types.
def premier_coup(pol, essais=8):
    """Ensemble des premiers coups possibles du type (singleton s'il est deterministe)."""
    return sorted({pol([]) for _ in range(essais)})

def attaque_apres_milieu_isole(pol, essais=8):
    """Le type attaque-t-il certainement apres un Milieu isole (historique [1]) ?"""
    return all(pol([1]) == 2 for _ in range(essais))

def reponse_depend_des_coups(pol):
    """La reponse varie-t-elle avec le dernier coup de l'agent (sonde a un tour) ?"""
    return len({pol([a]) for a in range(N_ACTIONS)}) > 1

def stochastique(pol, essais=16):
    return len({pol([0]) for _ in range(essais)}) > 1

props = {}
for nom, (_, pol) in TYPES.items():
    props[nom] = {
        'premier': premier_coup(pol),
        'attaque_milieu': attaque_apres_milieu_isole(pol),
        'reactif': reponse_depend_des_coups(pol),
        'stochastique': stochastique(pol),
    }
print(f"{'type':5s} {'premier coup':24s} {'apres Milieu':13s} {'reactif':9s} stochastique")
for nom, p in props.items():
    prem = ' ou '.join(ACTIONS[a] for a in p['premier'])
    print(f"{nom:5s} {prem:24s} {str(p['attaque_milieu']):13s} {str(p['reactif']):9s} {p['stochastique']}")

type  premier coup             apres Milieu  reactif   stochastique
t1    Attaquer                 True          True      False
t2    Securiser                False         True      False
t3    Attaquer                 True          False     False
t4    Securiser                True          True      False
t5    Securiser ou Milieu ou Attaquer False         True      True
t6    Attaquer                 True          True      False
t7    Securiser                False         False     False
t8    Securiser                False         False     False


In [4]:
# --- L'histoire observee h : 12 tours, l'agent a joue Milieu aux tours 8 et 12,
#     Securiser sinon ; l'adversaire n'a JAMAIS attaque pendant h.
HISTOIRE_AGENT = [0]*7 + [1] + [0]*3 + [1]

def deroule(pol, hist_agent):
    """Actions du type 'pol' face a la sequence donnee (tours simultanes : reponse retardee)."""
    faits = []
    for t in range(len(hist_agent)):
        faits.append(pol(hist_agent[:t]))
    return faits

def cible_de_h():
    return [0] * len(HISTOIRE_AGENT)

def compatible_avec_h(nom, essais=8):
    """Le type peut-il produire h exactement (deterministe : reproduction ; stochastique : possible) ?"""
    _, pol = TYPES[nom]
    if not props[nom]['stochastique']:
        return deroule(pol, HISTOIRE_AGENT) == cible_de_h()
    return any(deroule(pol, HISTOIRE_AGENT) == cible_de_h() for _ in range(essais))

# --- La base de croyances B : cinq atomes, chacun avec sa provenance.
#     b1-b3, b5 : sondes structurelles (a priori formes sur du jeu generique).
#     b4 : donnee de l'histoire h observeee (contrainte la plus lieee aux faits).
SENS = {
    'b1': {n for n in NOMS_TYPES if props[n]['reactif']},
    'b2': {n for n in NOMS_TYPES if props[n]['premier'] != [2]},
    'b3': {n for n in NOMS_TYPES if not props[n]['attaque_milieu']},
    'b4': {n for n in NOMS_TYPES if compatible_avec_h(n)},
    'b5': {n for n in NOMS_TYPES if not props[n]['stochastique']},
}
DESC = {
    'b1': 'sa reponse depend de mes coups',
    'b2': "son premier coup n'est pas toujours Attaquer",
    'b3': "il n'attaque jamais apres un Milieu isole",
    'b4': 'il est compatible avec h (12 tours sans attaque)',
    'b5': 'il est deterministe',
}

# --- Encodage Z3 : une variable booleenne par type ; atome = disjonction de ses types.
def z3_atome(code_atome):
    return z3.Or([z3.Bool(t) for t in sorted(SENS[code_atome])])

def z3_types(ensemble):
    if not ensemble:
        return z3.BoolVal(False)
    return z3.Or([z3.Bool(t) for t in sorted(ensemble)])

def z3_type_unique():
    """Le type cache est unique : une et une seule variable type est vraie."""
    return z3.PbEq([(z3.Bool(t), 1) for t in NOMS_TYPES], 1)

def types_admis(constraints):
    """Types rendus possibles par la conjonction : un controle Z3 par type."""
    admis = set()
    for n in NOMS_TYPES:
        s = z3.Solver()
        s.add(constraints)
        s.add(z3_type_unique())
        s.add(z3.Bool(n))
        if s.check() == z3.sat:
            admis.add(n)
    return admis

print("Base de croyances initiale B (semantique mesuree sur les politiques) :")
for code_a in SENS:
    print(f"  {code_a} : {DESC[code_a]}")
    print(f"       types : {sorted(SENS[code_a])}")

sat_B = z3.Solver().check([z3_atome(a) for a in SENS] + [z3_type_unique()])
types_B = types_admis([z3_atome(a) for a in SENS])
print(f"\nB est coherente (Z3 : {sat_B}) ; types admis : {sorted(types_B)}")

Base de croyances initiale B (semantique mesuree sur les politiques) :
  b1 : sa reponse depend de mes coups
       types : ['t1', 't2', 't4', 't5', 't6']
  b2 : son premier coup n'est pas toujours Attaquer
       types : ['t2', 't4', 't5', 't7', 't8']
  b3 : il n'attaque jamais apres un Milieu isole
       types : ['t2', 't5', 't7', 't8']
  b4 : il est compatible avec h (12 tours sans attaque)
       types : ['t2']
  b5 : il est deterministe
       types : ['t1', 't2', 't3', 't4', 't6', 't7', 't8']

B est coherente (Z3 : sat) ; types admis : ['t2']


### Lecture du banc

Chaque atome pris seul est raisonnable ; leur conjonction est radicale : `B` n'admet plus que
**le punisseur `t2`**. L'atome décisif est `b4` -- seul `t2` reproduit exactement les douze tours
sans attaque de `h`. Et c'est précisément là que le banc est piégé : le vrai type, `t8`, aurait
*attaqué* au tour 9 de `h` (en réponse au Milieu du tour 8) -- l'histoire observée n'était
compatible avec lui que parce que sa phase de prudence et le calendrier des Milieu de l'agent
ne se sont jamais croisés. La base est cohérente, mesurée... et fausse. C'est la situation que
la Section 2 va faire éclater.

Deux remarques de méthode. Les sémantiques des atomes ne sont pas déclarées : elles sont
**mesurées** sur les politiques (le tableau des sondes de la cellule précédente). Et `b3` --
« il n'attaque jamais après un Milieu *isolé* » -- garde `t8` dans son champ : le bluffeur
retardé n'attaque pas sur un Milieu isolé (phase de prudence), seulement sur un Milieu qui
s'inscrit dans une histoire établie. Cette nuance de sonde fera toute la différence au moment
de l'incision.

## Exercice 1 : énumérer les noyaux d'une base de croyances réseau

Une admin porte six croyances sur un attaquant inconnu. Les types possibles sont `A1..A8`.
La base :

| Atome | Croyance | Types compatibles |
|---|---|---|
| `c1` | l'attaquant ne fait que de la reconnaissance | {A1, A2, A3, A5, A6} |
| `c2` | l'attaquant agit seulement la nuit | {A1, A3, A4, A7} |
| `c3` | l'attaquant évite les serveurs de production | {A2, A5, A6, A8} |
| `c4` | l'attaquant n'a pas de credential valide | {A1, A2, A5, A6, A7, A8} |
| `c5` | l'attaquant est interne au réseau | {A2, A4, A5, A6} |
| `c6` | l'attaquant ne persiste pas au redémarrage | {A1, A3, A5, A7, A8} |

L'évidence du jour : l'attaquant vient d'utiliser un **credential valide** -- les types
compatibles avec cette évidence sont {A3, A4}. Un **noyau** de la base (pour cette évidence)
est un sous-ensemble *minimal* d'atomes dont la conjonction avec l'évidence est insatisfaisante.

**Travail demandé** : énumérer tous les noyaux de cette base pour cette évidence, sous forme
de liste de listes d'atomes triées (`[['c1', 'c4'], ...]` par exemple). Vérifier la minimalité
de chacun.

In [5]:
# --- Exercice 1 : a completer par l'etudiant ---------------------------------
C_SENSE = {
    'c1': {'A1', 'A2', 'A3', 'A5', 'A6'},
    'c2': {'A1', 'A3', 'A4', 'A7'},
    'c3': {'A2', 'A5', 'A6', 'A8'},
    'c4': {'A1', 'A2', 'A5', 'A6', 'A7', 'A8'},
    'c5': {'A2', 'A4', 'A5', 'A6'},
    'c6': {'A1', 'A3', 'A5', 'A7', 'A8'},
}
C_EVIDENCE = {'A3', 'A4'}

def noyaux_exercice(sense, evidence):
    """Renvoie la liste des noyaux : sous-ensembles minimaux d'atomes incompatibles avec l'evidence.

    Indice 1 : un sous-ensemble K d'atomes est incompatible si l'intersection des sense(K)
    avec l'ensemble evidence est vide.
    Indice 2 : K est minimal si aucun sous-ensemble strict de K n'est deja incompatible.
    Indice 3 : itertools.combinations(sense, k) enumere les sous-ensembles de taille k ;
    monter de k = 1 a len(sense). Des noyaux de tailles differentes peuvent coexister.
    """
    # TODO etudiant : construire et retourner la liste des noyaux (liste de listes triees).
    return None  # TODO etudiant

noyaux_reponse = noyaux_exercice(C_SENSE, C_EVIDENCE)
if noyaux_reponse is None:
    print("Exercice a completer : implementer noyaux_exercice puis relancer cette cellule.")
else:
    for k in noyaux_reponse:
        print("noyau :", sorted(k))
    print(f"{len(noyaux_reponse)} noyau(x)")

Exercice a completer : implementer noyaux_exercice puis relancer cette cellule.


## Section 2 -- L'évidence contredit : noyaux et fonction d'incision

Au tour 13, l'agent joue *Milieu* ; l'adversaire réplique **Attaquer**. C'est l'évidence
`phi` : *il attaque en réponse à mon Milieu* -- mesurée sur les huit politiques face à
l'historique réel `h` (une évidence est liée aux faits ; les atomes de `B`, eux, venaient de
sondes génériques). La base devient incohérente. La théorie AGM ne dit pas « tout jeter » :
elle **contracte**.

La contraction par noyaux (Alchourron & Makinson) procède en deux temps :

1. **énumérer les noyaux** de `B` pour `phi` : les sous-ensembles *minimaux* d'atomes dont
   la conjonction avec `phi` est insatisfaisable -- Z3 tranche de chaque candidat ;
2. **choisir une incision** : retirer au moins un atome de *chaque* noyau, en sacrifiant
   en priorité les atomes les *moins enracinés*.

L'enracinement (`epistemic entrenchment`) n'est pas déclaré ici : il est **mesuré**. Chaque
atome est évalué sur une seconde histoire `h2`, tenue à part, où le vrai type finit par se
révéler : un atome est falsifié sur un tour de `h2` s'il exclut *tous* les types capables
d'expliquer l'action observée. L'atome qui prédit le mieux ce qui s'est réellement passé est
le plus enraciné -- le dernier qu'on accepte de sacrifier.

In [6]:
# --- Enumeration des noyaux par taille croissante, chaque candidat juge par Z3
def noyaux(sense, evidence):
    """Noyaux de la base pour l'evidence : candidats par taille, minimaleite par inclusion."""
    atomes = sorted(sense)
    trouves: List[frozenset] = []
    for k in range(1, len(atomes) + 1):
        for combo in itertools.combinations(atomes, k):
            contraintes = [z3.Or([z3.Bool(t) for t in sorted(sense[a])]) for a in combo]
            contraintes.append(z3_types(evidence))
            contraintes.append(z3_type_unique())
            if z3.Solver().check(contraintes) == z3.unsat:
                if not any(set(s) <= set(combo) for s in trouves):
                    trouves.append(frozenset(combo))
    return [sorted(k) for k in trouves]

PHI = {n for n in NOMS_TYPES
       if TYPES[n][1](HISTOIRE_AGENT) == 2 and not props[n]['stochastique']}
print(f"Evidence phi = 'il attaque apres mon Milieu (tour 13)' : types {sorted(PHI)}")

noyaux_B = noyaux(SENS, PHI)
print(f"\nNoyaux de B pour phi ({len(noyaux_B)}) :")
for k in noyaux_B:
    print(f"  {k}")
    for a in k:
        print(f"      {a} : {DESC[a]}")

# Controle croise : le core Z3 (assert_and_track) retrouve-t-il un de ces noyaux ?
s = z3.Solver()
hyp = {}
for a in sorted(SENS):
    hyp[a] = z3.Bool(f"hyp_{a}")
    s.assert_and_track(z3.Or([z3.Bool(t) for t in sorted(SENS[a])]), hyp[a])
s.add(z3_types(PHI))
s.add(z3_type_unique())
res = s.check(list(hyp.values()))
core = sorted(str(c).replace("hyp_", "") for c in s.unsat_core())
print(f"\nControle croise Z3 (assert_and_track) : {res}, core = {core}")
print(f"Le core est-il l'un des noyaux enumerees ? {core in noyaux_B}")

Evidence phi = 'il attaque apres mon Milieu (tour 13)' : types ['t1', 't3', 't4', 't6', 't8']

Noyaux de B pour phi (2) :
  ['b4']
      b4 : il est compatible avec h (12 tours sans attaque)
  ['b1', 'b3']
      b1 : sa reponse depend de mes coups
      b3 : il n'attaque jamais apres un Milieu isole

Controle croise Z3 (assert_and_track) : unsat, core = ['b4']
Le core est-il l'un des noyaux enumerees ? True


In [7]:
# --- Enracinement MESURE : valeur predictive de chaque atome sur l'histoire tenue a part h2.
#     h2 : 30 tours contre le VRAI type (bluffeur retarde), l'agent alternant selon un schema fixe.
SCHEMA_AGENT_H2 = [0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0,
                   1, 0, 1, 1, 0, 1, 1, 0, 1, 1]

def actions_d_un_type(nom_type, hist_agent):
    """Actions du type sur hist_agent, RNG reseedee par type pour la reproductibilite."""
    global RNG
    RNG = np.random.default_rng(seed=20261010 + 97 * NOMS_TYPES.index(nom_type))
    _, pol = TYPES[nom_type]
    return deroule(pol, hist_agent)

VRAI_TYPE = 't8'
ACTIONS_H2 = actions_d_un_type(VRAI_TYPE, SCHEMA_AGENT_H2)

def types_expliquant(tour, action_vue):
    """Types capables de produire 'action_vue' au tour donne de h2."""
    expl = set()
    for n in NOMS_TYPES:
        if actions_d_un_type(n, SCHEMA_AGENT_H2)[tour] == action_vue:
            expl.add(n)
    return expl

enracinement = {}
for a in SENS:
    falsifie = sum(
        1 for tour, act in enumerate(ACTIONS_H2)
        if (expl := types_expliquant(tour, act)) and not (SENS[a] & expl)
    )
    enracinement[a] = 1.0 - falsifie / len(ACTIONS_H2)

print("Enracinement mesure sur h2 (taux de tours correctement expliques) :")
for a, score in sorted(enracinement.items(), key=lambda kv: (kv[1], -len(SENS[kv[0]]))):
    print(f"  {a} = {score:.3f}   {DESC[a]}")

# Ordre croissant : score predictif d'abord, puis informativite (exclut plus de types = plus enracine)
ORDRE = [a for a, _ in sorted(enracinement.items(),
                              key=lambda kv: (kv[1], -len(SENS[kv[0]])))]
print(f"\nOrdre croissant d'enracinement (moins enracine a gauche) : {' < '.join(ORDRE)}")

Enracinement mesure sur h2 (taux de tours correctement expliques) :
  b4 = 0.500   il est compatible avec h (12 tours sans attaque)
  b5 = 1.000   il est deterministe
  b1 = 1.000   sa reponse depend de mes coups
  b2 = 1.000   son premier coup n'est pas toujours Attaquer
  b3 = 1.000   il n'attaque jamais apres un Milieu isole

Ordre croissant d'enracinement (moins enracine a gauche) : b4 < b5 < b1 < b2 < b3


In [8]:
# --- Incision sous enracinement, puis revision : B * phi = (B sans l'incision) + phi
def incision(noyaux_liste, ordre):
    """Coupe au moins un atome par noyau, le moins enracine possible."""
    coupee = set()
    for noy in noyaux_liste:
        if not (set(noy) & coupee):
            coupee.add(min(noy, key=ordre.index))
    return coupee

INCISION = incision(noyaux_B, ORDRE)
ATOMES_SURVIVANTS = [a for a in sorted(SENS) if a not in INCISION]
types_B_revise = types_admis([z3_atome(a) for a in ATOMES_SURVIVANTS] + [z3_types(PHI)])

print(f"Incision (atomes sacrifies)       : {sorted(INCISION)}")
print(f"Atomes qui survivent              : {ATOMES_SURVIVANTS}")
print(f"Types admis par B  (avant)        : {sorted(types_B)}")
print(f"Types admis par B * phi (apres)   : {sorted(types_B_revise)}")
print(f"Le type vrai ({VRAI_TYPE}) est-il admis apres revision ? {VRAI_TYPE in types_B_revise}")

Incision (atomes sacrifies)       : ['b1', 'b4']
Atomes qui survivent              : ['b2', 'b3', 'b5']
Types admis par B  (avant)        : ['t2']
Types admis par B * phi (apres)   : ['t8']
Le type vrai (t8) est-il admis apres revision ? True


### Lecture des noyaux et de l'incision

Deux noyaux, de formes différentes -- c'est ce qui rend l'incision intéressante :

- `['b4']` est un noyau **singleton** : la compatibilité avec `h`, à elle seule, exclut tous
  les types de `phi` ;
- `['b1', 'b3']` est un noyau **par paire** : « il réagit à mes coups » et « il n'attaque jamais
  après un Milieu isolé » pris ensemble n'admettent plus aucun type qui attaque après Milieu --
  alors que chacun seul en admet encore (`t1`/`t4`/`t6` pour `b1`, `t8` pour `b3`).

Le contrôle croisé le montre proprement : le *core* de Z3 (`assert_and_track`) rend `['b4']` --
un noyau -- mais un seul ; l'énumération par taille croissante rend les deux. Le core répond
« voilà un conflit », l'énumération répond « voilà *tous* les conflits minimaux ».

L'enracinement mesuré ne départage que `b4` : falsifié sur les quinze tours d'attaque de `h2`
(score 0,500), quand les quatre autres atomes expliquent chaque tour (1,000). Les ex æquo se
départagent par l'informativité (un atome qui exclut plus de types est plus enraciné), d'où
l'ordre `b4 < b5 < b1 < b2 < b3`. L'incision tombe alors : `b4` (seul membre du premier noyau)
et, dans la paire, `b1` -- moins enraciné que `b3`. La révision `B * phi` admet exactement
**`t8`, le type vrai** : en sacrifiant la lecture de l'histoire et l'hypothèse de réactivité,
l'agent garde « n'attaque jamais après un Milieu isolé » et « déterministe » -- deux atomes
vrais de `t8` -- et l'évidence affine le reste. Le changement est chirurgical : rien de ce qui
survit n'était falsifié par `h2`.

In [9]:
# --- Controles negatifs : (a) evidence coherente => expansion pure ; (b) noyau force unique
# (a) une evidence compatible avec B : 'son premier coup n'est pas toujours Attaquer'
PSI = {n for n in NOMS_TYPES if props[n]['premier'] != [2]}
noyaux_psi = noyaux(SENS, PSI)
types_expansion = types_admis([z3_atome(a) for a in sorted(SENS)] + [z3_types(PSI)])
print(f"(a) evidence coherente (types {sorted(PSI)}) : noyaux = {noyaux_psi}")
print(f"    incision vide -> la revision DEGENERE en expansion (B + psi), zero perte")
print(f"    types apres expansion : {sorted(types_expansion)} "
      f"(inclus dans les types de B : {types_expansion <= types_B})")

# (b) une evidence qui force la chute de l'atome le PLUS enracine :
#     'son premier coup est certainement Attaquer'
THETA = {n for n in NOMS_TYPES if props[n]['premier'] == [2]}
noyaux_theta = noyaux(SENS, THETA)
inc_theta = incision(noyaux_theta, ORDRE)
plus_enracine = ORDRE[-1]
print(f"\n(b) evidence 'il attaque d'emblee, certainement' (types {sorted(THETA)})")
print(f"    noyaux = {noyaux_theta}")
print(f"    incision imposee : {sorted(inc_theta)}")
print(f"    l'atome le plus enracine ({plus_enracine}) tombe-t-il ? {plus_enracine in inc_theta}")
print("    -> quand aucun noyau alternatif n'existe, l'ordre ne protege personne : mesure, pas negocie")

(a) evidence coherente (types ['t2', 't4', 't5', 't7', 't8']) : noyaux = []
    incision vide -> la revision DEGENERE en expansion (B + psi), zero perte
    types apres expansion : ['t2'] (inclus dans les types de B : True)



(b) evidence 'il attaque d'emblee, certainement' (types ['t1', 't3', 't6'])
    noyaux = [['b2'], ['b3'], ['b4']]
    incision imposee : ['b2', 'b3', 'b4']
    l'atome le plus enracine (b3) tombe-t-il ? True
    -> quand aucun noyau alternatif n'existe, l'ordre ne protege personne : mesure, pas negocie


### Lecture des contrôles négatifs

**(a) L'évidence cohérente ne coûte rien.** Aucun noyau pour `psi`, incision vide : la
« révision » dégénère en expansion, les types admis ne changent pas (`{t2}`). C'est la
propriété de parcimonie du changement de croyance : confirmer n'est pas réviser -- mesuré,
pas décrété.

**(b) L'ordre d'enracinement n'est pas un veto.** Face à l'évidence « il attaque d'emblée,
certainement », trois noyaux singletons surgissent (`b2`, `b3`, `b4`) : aucun choix n'existe,
et l'atome le *plus* enraciné (`b3`) tombe avec les autres. L'enracinement hiérarchise les
sacrifices quand des alternatives existent ; il ne protège jamais une croyance que chaque
noyau accuse séparément. La mesure le dit crûment : c'est un ordre de coût, pas un droit de
veto.

## Exercice 2 : implémenter la fonction d'incision sous ordre d'enracinement

La fonction d'incision reçoit la liste des noyaux et un ordre d'enracinement (du moins au plus
enraciné). Elle retourne un ensemble d'atomes qui **touche chaque noyau au moins une fois**,
en privilégiant les atomes les moins enracinés. La batterie de vérification contrôle les deux
propriétés attendues : couverture complète des noyaux, et *finesse* (ne couper que ce qu'aucun
choix meilleur ne permettait d'épargner).

In [10]:
# --- Exercice 2 : a completer par l'etudiant ---------------------------------
def incision_exercice(noyaux_liste, ordre):
    """Retourne l'ensemble (set) des atomes a sacrifier.

    Indice 1 : parcourir les noyaux ; si le noyau n'est pas deja touche, couper son atome
    le moins enracine (d'indice minimal dans 'ordre').
    Indice 2 : un atome coupe peut toucher plusieurs noyaux a la fois -- c'est tout l'interet.
    """
    # TODO etudiant : construire et retourner l'ensemble d'incision.
    return None  # TODO etudiant

# --- Batterie de verification (executee seulement si l'exercice est complete) ---
BATTERIE_INCISION = [
    ([['a', 'b']], ['a', 'b'], {'a'}),
    ([['a', 'b'], ['b', 'c']], ['c', 'b', 'a'], {'b'}),
    ([['a', 'b'], ['a', 'c']], ['c', 'b', 'a'], {'a'}),
    ([['a'], ['b', 'c']], ['c', 'b', 'a'], {'a', 'c'}),
    ([['a', 'b', 'c']], ['c', 'b', 'a'], {'c'}),
]
if incision_exercice([['a', 'b']], ['a', 'b']) is None:
    print("Exercice a completer : implementer incision_exercice puis relancer cette cellule.")
else:
    exacts = 0
    for noys, ordre, attendu in BATTERIE_INCISION:
        obtenu = set(incision_exercice(noys, ordre))
        couvre = all(set(n) & obtenu for n in noys)
        statut = 'PASS' if couvre and obtenu == attendu else ('COUVRANT' if couvre else 'ECHEC')
        exacts += statut == 'PASS'
        print(f"noyaux={noys} ordre={ordre} -> {sorted(obtenu)} attendu={sorted(attendu)} [{statut}]")
    print(f"\n{exacts}/{len(BATTERIE_INCISION)} scenarios exacts (couverture + finesse)")

Exercice a completer : implementer incision_exercice puis relancer cette cellule.


## Section 3 -- Le témoin : réviser coûte moins que réinitialiser ou refuser

Le vrai type caché est `t8` (bluffeur retardé). À partir du tour 13 -- juste après l'évidence
qui a déclenché la révision -- l'agent joue 200 tours selon trois attitudes épistémiques :

- **révisé** (AGM) : meilleure réponse espérée contre les types admis par `B * phi` ;
- **reset naïf** : il ne croit plus rien, meilleure réponse contre les huit types à égalité ;
- **refus conservateur** : il garde `B` intact et fait comme si de rien n'était.

La meilleure réponse maximise le gain espéré contre la crédence uniforme sur les types encore
crus, l'action adverse du tour étant prédite par les politiques réelles (tours simultanés :
la prédiction ne dépend pas de l'action courante de l'agent, l'influence est décalée d'un tour).
Le tableau final est le témoin mesuré : ce que coûte chaque attitude, en unités du jeu.

In [11]:
# --- Le temoin : 200 tours contre le vrai type t8, trois attitudes epistemiques
def action_predite(nom_type, hist):
    """Action adverse du tour suivant, predite par la politique (RNG figee par type)."""
    global RNG
    RNG = np.random.default_rng(seed=20261010 + 97 * NOMS_TYPES.index(nom_type))
    _, pol = TYPES[nom_type]
    return pol(hist)

def meilleure_reponse(types_crus, hist):
    """Action de l'agent maximisant le gain espere contre la credence uniforme sur les types crus."""
    if not types_crus:
        return 0  # croyance vide : repli defensif
    gains = np.zeros(N_ACTIONS)
    for n in sorted(types_crus):
        b = action_predite(n, hist)
        gains += GAIN_AGENT[:, b] / len(types_crus)
    return int(np.argmax(gains))

def partie(attitude, tours=200, coupure=13):
    """Joue la partie contre le VRAI type t8 (deterministe : aucune RNG n'entre en jeu)."""
    _, pol_vrai = TYPES[VRAI_TYPE]
    hist, gain_total = [], 0.0
    for t in range(tours):
        if t < coupure:
            crus = set(types_B)
        elif attitude == 'revise':
            crus = set(types_B_revise)
        elif attitude == 'reset':
            crus = set(NOMS_TYPES)
        else:  # refus
            crus = set(types_B)
        a = meilleure_reponse(crus, hist)
        b = pol_vrai(hist)
        gain_total += GAIN_AGENT[a, b]
        hist.append(a)
    return gain_total

RESULTATS = {att: partie(att) for att in ('revise', 'reset', 'refus')}
print("Gain cumule sur 200 tours contre le vrai type t8 (bluffeur retarde) :")
for att, g in RESULTATS.items():
    print(f"  {att:8s} : {g:+8.1f}")
print(f"\nValeur du changement minimal (revise - reset) : {RESULTATS['revise'] - RESULTATS['reset']:+.1f}")
print(f"Cout du refus d'evidence (revise - refus)      : {RESULTATS['revise'] - RESULTATS['refus']:+.1f}")

Gain cumule sur 200 tours contre le vrai type t8 (bluffeur retarde) :
  revise   :   +450.0
  reset    :    -15.0
  refus    :   -765.0

Valeur du changement minimal (revise - reset) : +465.0
Cout du refus d'evidence (revise - refus)      : +1215.0


### Lecture du témoin

Les trois attitudes, 200 tours contre le vrai bluffeur retardé :

| Attitude | Gain cumulé | Ce qui s'est passé |
|---|---|---|
| révisé (AGM) | **+450,0** | croit `t8` exactement : alterne Milieu (+3 sur la prudence) et Attaquer (+2 sur l'attaque) |
| reset naïf | **-15,0** | croit les 8 types à égalité : joue couvert, ne s'engage jamais |
| refus | **-765,0** | croit toujours `t2` : prédit la prudence, joue Milieu -- et mange -4 par tour |

Le refus est le piège auto-confirmant : croire le punisseur fait jouer Milieu (la meilleure
réponse à sa prudence), et ce Milieu répété déclenche l'attaque du bluffeur réel, que la
croyance fausse continue d'ignorer. Le reset, lui, ne se fait pas piéger -- mais il jette
l'information : il ne sait pas non plus *exploiter*. La révision gardait deux atomes vrais de
`t8`, et c'est exactement ce qu'ils valent : **la valeur du changement minimal est de +465,0**
(révisé moins reset) et **le coût du refus d'évidence de +1215,0** (révisé moins refus) --
refuser de réviser coûte ici plus de deux fois la valeur de toute l'information accumulée.

## Section 4 -- Les propriétés de la révision : la batterie de tests

La révision par noyaux satisfait les postulats AGM *par construction* -- c'est un théorème.
Mais ce carnet ne demande pas de le croire : la batterie ci-dessous **tire des bases
aléatoires**, calcule la révision par noyaux et incision, et **contrôle chaque propriété sur
chaque cas**. Un échec s'afficherait sous son nom, avec son compteur.

In [12]:
# --- Batterie : bases aleatoires, revision par noyaux, proprietes verifiees une a une
def revision_noyaux(sense, evidence, ordre):
    noys = noyaux(sense, evidence)
    coupes = incision(noys, ordre)
    admis = set(NOMS_TYPES)
    for a, ens in sense.items():
        if a not in coupes:
            admis &= ens
    return admis & evidence

PROPRIETES = ['succes', 'inclusion', 'vacuite', 'coherence', 'extensionnalite',
              'sous-expansion', 'super-expansion']
compteurs = {p: 0 for p in PROPRIETES}
cas = 0
for essai in range(20):
    n_at = int(RNG.integers(3, 7))
    atomes_alea = [f"x{i}" for i in range(n_at)]
    sense_alea = {a: set(RNG.choice(NOMS_TYPES, size=int(RNG.integers(3, 8)),
                                    replace=False).tolist()) for a in atomes_alea}
    phi_alea = set(RNG.choice(NOMS_TYPES, size=int(RNG.integers(1, 4)),
                              replace=False).tolist())
    ordre_alea = list(RNG.permutation(atomes_alea))
    admis_B = set(NOMS_TYPES)
    for ens in sense_alea.values():
        admis_B &= ens

    rev = revision_noyaux(sense_alea, phi_alea, ordre_alea)
    cas += 1
    compteurs['succes'] += rev <= phi_alea
    compteurs['inclusion'] += rev <= (admis_B | phi_alea)
    compteurs['vacuite'] += (rev == admis_B & phi_alea) if (admis_B & phi_alea) else True
    compteurs['coherence'] += bool(rev)
    compteurs['extensionnalite'] += rev == revision_noyaux(sense_alea, set(phi_alea), ordre_alea)
    psi_alea = set(RNG.choice(NOMS_TYPES, size=int(RNG.integers(1, 4)),
                              replace=False).tolist())
    gauche = revision_noyaux(sense_alea, phi_alea & psi_alea, ordre_alea)
    compteurs['sous-expansion'] += (gauche == rev & psi_alea) if (rev & psi_alea) else True
    compteurs['super-expansion'] += gauche <= (rev & psi_alea) if (rev & psi_alea) else True

print(f"Batterie : {cas} bases aleatoires (3 a 6 atomes), revision par noyaux + incision")
print(f"{'propriete':16s} passes")
for p in PROPRIETES:
    print(f"{p:16s} {compteurs[p]}/{cas}")

Batterie : 20 bases aleatoires (3 a 6 atomes), revision par noyaux + incision
propriete        passes
succes           20/20
inclusion        20/20
vacuite          20/20
coherence        20/20
extensionnalite  20/20
sous-expansion   20/20
super-expansion  20/20


### Lecture de la batterie

Sept propriétés, vingt bases aléatoires, **20/20 partout**. Deux lectures :

- la **cohérence** (le résultat n'est jamais vide) tient mécaniquement à la frappe complète
  des noyaux : si les atomes survivants excluaient encore tout `phi`, ils contiendraient un
  noyau intact -- contredisant l'incision ;
- la **vacuité** et la **sous-expansion** sont conditionnelles par construction (elles ne
  s'engagent que quand l'évidence est cohérente avec la base) : c'est exactement le contenu
  du postulat -- une évidence cohérente ne doit rien coûter.

Ce sont les postulats AGM de base sous forme ensembliste finie. La révision *itérée*
(Darwiche-Pearl : réviser deux fois n'est pas réviser la conjonction) n'est pas testée ici --
l'incision gloutonne n'engage rien sur elle, et c'est une limite assumée de ce carnet.

## Exercice 3 : mesurer le témoin contre un autre type vrai

Rejouer la partie de la Section 3 contre le type `t4` (bluffeur immédiat, sans phase de
prudence initiale) : le classement des trois attitudes change-t-il ? En particulier, le
**refus** coûte-t-il plus ou moins cher face à un bluffeur qui se révèle dès le premier
Milieu ? Retourner le dictionnaire `{'revise': ..., 'reset': ..., 'refus': ...}` des gains
cumulés.

In [13]:
# --- Exercice 3 : a completer par l'etudiant ---------------------------------
def partie_contre_t4(attitude, tours=200, coupure=13):
    """Meme partie que la Section 3, mais le vrai type est t4 (bluffeur immediat).

    Indice 1 : reutiliser meilleure_reponse -- elle ne depend pas du type vrai.
    Indice 2 : seul le calcul de l'action adverse change (pol_bluffeur au lieu de t8) ;
    t4 est deterministe, aucune RNG n'entre en jeu.
    Indice 3 : les croyances (types_B, types_B_revise) sont deja calculees dans les
    sections precedentes -- ne pas les recalculer.
    """
    # TODO etudiant : construire et retourner le gain cumule de l'attitude demandee.
    return None  # TODO etudiant

rep3 = {att: partie_contre_t4(att) for att in ('revise', 'reset', 'refus')}
if any(v is None for v in rep3.values()):
    print("Exercice a completer : implementer partie_contre_t4 puis relancer cette cellule.")
else:
    for att, g in rep3.items():
        print(f"  {att:8s} : {g:+8.1f}")

Exercice a completer : implementer partie_contre_t4 puis relancer cette cellule.


## Conclusion -- changer d'avis en perdant le moins possible

Ce carnet a attesté l'opération **réviser une croyance** sur un substrat indépendant de
l'attestation logique classique :

- la **première attestation** ([Tweety-04](../SymbolicAI/Tweety/Tweety-04-Belief-Revision-Python.ipynb))
  révisait des bases propositionnelles par les opérateurs CrMas de Tweety, sur JVM ;
- la **seconde** (celle-ci) révisait un modèle d'adversaire en jeu répété : consistance
  décidée par Z3, enracinement *mesuré* par valeur prédictive, postulats *testés* sur
  batterie aléatoire.

Les trois morceaux du témoin sont dans les sorties : la trace de révision (noyaux, incision,
survie des atomes), le coût mesuré des attitudes rivales (refus, reset), et le contrôle
négatif (l'évidence cohérente ne coûte rien). La limite assumée : l'espace de types est fini
et petit, et l'incision gloutonne n'engage rien sur la révision itérée (Darwiche-Pearl) --
l'étendre à des croyances structurées sur espaces continus reste hors de ce carnet.

Pour le tableau des opérations ([#12204](https://github.com/jsboige/CoursIA/issues/12204)),
la forme du témoin est maintenant établie deux fois, sur des substrats indépendants par
l'outil (JVM/Tweety contre Z3/jeux) et par le stimulus (bases propositionnelles contre
modèles d'adversaire) -- la statuation d'admission reste au processus de l'EPIC.

**Navigation** : [<< 13d-Optimistic-CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) | [Index](README.md)